# Motion Transfer Backend (T4 Optimized)
Mimic-style pipeline with FastAPI

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate xformers einops opencv-python pillow imageio imageio-ffmpeg
!pip install -q mediapipe fastapi uvicorn pyngrok realesrgan

In [ ]:
import os, cv2, torch, shutil
import numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile
from fastapi.responses import FileResponse

DEVICE='cuda'
DTYPE=torch.float16

os.makedirs('input', exist_ok=True)
os.makedirs('frames', exist_ok=True)
os.makedirs('output', exist_ok=True)

In [ ]:
def extract_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    i=0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        path=f'frames/{i:04d}.png'
        cv2.imwrite(path, frame)
        frames.append(path)
        i+=1
    cap.release()
    return frames

In [ ]:
import mediapipe as mp
mp_pose = mp.solutions.pose.Pose(model_complexity=1)

def extract_pose(image_path):
    img=cv2.imread(image_path)
    img_rgb=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res=mp_pose.process(img_rgb)
    canvas=np.zeros_like(img)
    if res.pose_landmarks:
        for lm in res.pose_landmarks.landmark:
            x=int(lm.x*img.shape[1])
            y=int(lm.y*img.shape[0])
            cv2.circle(canvas,(x,y),3,(255,255,255),-1)
    return canvas

In [ ]:
def chunk_frames(frames, size=24, overlap=4):
    chunks=[]
    i=0
    while i<len(frames):
        chunks.append(frames[i:i+size])
        i+=size-overlap
    return chunks

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5', torch_dtype=DTYPE
).to(DEVICE)

pipe.enable_xformers_memory_efficient_attention()
pipe.enable_model_cpu_offload()

In [ ]:
def generate_chunk(chunk, char_img_path):
    outputs=[]
    char_img=Image.open(char_img_path).resize((512,512))
    for i,f in enumerate(chunk):
        pose=extract_pose(f)
        pose_img=Image.fromarray(pose).resize((512,512))
        img=pipe(prompt='realistic dancing human, skin texture, cinematic', image=char_img, strength=0.6, guidance_scale=3, num_inference_steps=14).images[0]
        out=f'output/frame_{len(outputs):04d}.png'
        img.save(out)
        outputs.append(out)
    return outputs

In [ ]:
def frames_to_video(frames, out='output/raw.mp4', fps=15):
    frame=cv2.imread(frames[0])
    h,w,_=frame.shape
    vid=cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))
    for f in frames:
        vid.write(cv2.imread(f))
    vid.release()
    return out

In [ ]:
def interpolate(inp, out='output/interp.mp4'):
    os.system(f"ffmpeg -y -i {inp} -vf minterpolate='fps=30' {out}")
    return out

In [ ]:
def upscale(inp, out='output/final.mp4'):
    os.system(f"ffmpeg -y -i {inp} -vf scale=1280:720 {out}")
    return out

In [ ]:
def process(image, video):
    frames=extract_frames(video)
    chunks=chunk_frames(frames)
    all_out=[]
    for c in chunks:
        all_out+=generate_chunk(c, image)
    raw=frames_to_video(all_out)
    inter=interpolate(raw)
    final=upscale(inter)
    return final

In [ ]:
app=FastAPI()

@app.post('/generate')
async def generate(character_image: UploadFile, reference_video: UploadFile):
    img_path='input/char.png'
    vid_path='input/ref.mp4'
    with open(img_path,'wb') as f:
        shutil.copyfileobj(character_image.file,f)
    with open(vid_path,'wb') as f:
        shutil.copyfileobj(reference_video.file,f)
    out=process(img_path, vid_path)
    return FileResponse(out, media_type='video/mp4')

In [ ]:
from pyngrok import ngrok
import uvicorn

url=ngrok.connect(8000)
print('URL:', url)

uvicorn.run(app, host='0.0.0.0', port=8000)